# Vocal Coach — Sprint 2 Dual-track Demo

Visualizes the per-song artifacts produced by Sprint 2 on the **Losing My Religion** UltraStar bundle:

1. **UltraStar note grid** (piano-roll of MIDI targets per syllable).
2. **Reference NanoPitch F0** vs **user NanoPitch F0** on the same time axis.
3. **Reference STARS** technique bands and **user STARS** technique bands (when available).
4. **Per-note `pct_in_tune`** + **highlight regions** from `scripts/analyze_performance.py`.

Run before opening this notebook:

```powershell
python scripts/import_ultrastar.py data/songs/losing-my-religion `
    --reference-vocal reference_vocal.mp3 --instrumental instrumental.mp3
python scripts/build_song.py data/songs/losing-my-religion
python scripts/analyze_performance.py data/songs/losing-my-religion `
    "data/Losing-My-Religion/Losing My Religion_user_vocals.wav" --perf-id take-1
```

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'vocal_coach').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from vocal_coach.reference import load_reference
from vocal_coach.schemas import (
    PerformanceAnalysis,
    PitchTrack,
    StarsTrack,
)
from vocal_coach.song import load_manifest

SONG_ID = 'losing-my-religion'
PERF_ID = 'take-1'
SONG_DIR = ROOT / 'data' / 'songs' / SONG_ID
PERF_DIR = SONG_DIR / 'performances' / PERF_ID

manifest = load_manifest(SONG_DIR)
reference = load_reference(SONG_DIR)
analysis_path = PERF_DIR / 'analysis.json'
assert analysis_path.is_file(), f'Run scripts/analyze_performance.py first; missing {analysis_path}'
analysis = PerformanceAnalysis.model_validate_json(analysis_path.read_text(encoding='utf-8'))

print(f'song_id            : {manifest.song_id}')
print(f'title              : {manifest.title}')
print(f'duration           : {manifest.duration_s:.2f}s')
print(f'#notes (chart)     : {len(reference.notes)}')
print(f'global_offset_s    : {analysis.global_offset_s:+.3f}')
print(f'#highlights        : {len(analysis.highlights.moments)}')
for m in analysis.highlights.moments:
    print(f'  - {m.type}: {m.title} ({m.start_s:.2f}-{m.end_s:.2f}s, score={m.score:.2f})')

## Reference vs user pitch

In [ ]:
def _load_pitch(rel_path):
    p = SONG_DIR / rel_path
    return PitchTrack.model_validate_json(p.read_text(encoding='utf-8')) if p.is_file() else None

pitch_ref = _load_pitch(analysis.pitch_ref_path) if analysis.pitch_ref_path else None
pitch_user = _load_pitch(analysis.pitch_user_path)

def _f0_to_midi(f0):
    midi = np.full_like(f0, np.nan, dtype=np.float64)
    voiced = f0 > 0
    midi[voiced] = 69.0 + 12.0 * np.log2(f0[voiced] / 440.0)
    return midi

def _xy_from_pitch(track):
    if track is None:
        return None, None
    t = np.array([f.time for f in track.frames])
    f0 = np.array([f.f0_hz for f in track.frames])
    return t, _f0_to_midi(f0)

tr, mr = _xy_from_pitch(pitch_ref)
tu, mu = _xy_from_pitch(pitch_user)
# Shift user trace into song time so it aligns with the chart.
if tu is not None:
    tu_song = tu - analysis.global_offset_s
else:
    tu_song = None

fig, ax = plt.subplots(figsize=(14, 4))
for note in reference.notes:
    ax.add_patch(Rectangle((note.start_s, note.midi_pitch - 0.45), note.end_s - note.start_s, 0.9, color='#cccccc', alpha=0.5))
if tr is not None:
    ax.plot(tr, mr, color='#5cd0a5', lw=0.8, label='reference NanoPitch')
if tu_song is not None:
    ax.plot(tu_song, mu, color='#6cb4ff', lw=0.8, label='user NanoPitch (in song time)')
ax.set_xlabel('song time (s)')
ax.set_ylabel('MIDI pitch')
ax.legend(loc='upper right')
ax.set_title(f'{manifest.title} — reference vs user pitch')
plt.show()

## Per-note `pct_in_tune` + highlights

Each bar is one UltraStar note in song-time, colored by `pct_in_tune` (green = clean, red = struggle). Highlight regions are overlaid as colored boxes.

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(14, 3))
for n, m in zip(reference.notes, analysis.notes):
    pct = m.pct_in_tune if m.pct_in_tune is not None else 0.0
    color = cm.RdYlGn(pct)
    ax.add_patch(Rectangle((n.start_s, 0), n.end_s - n.start_s, 1, color=color, alpha=0.85))
for moment in analysis.highlights.moments:
    color = {
        'best_pitch_phrase': '#5cd0a5',
        'pitch_struggle': '#e26e6e',
        'expressive_match': '#c98cf6',
        'expressive_moment': '#c98cf6',
        'missed_expression': '#f0b35b',
        'late_entrance': '#f0b35b',
    }.get(moment.type, '#888888')
    ax.add_patch(Rectangle((moment.start_s, 1.05), moment.end_s - moment.start_s, 0.3, color=color, alpha=0.9))
    ax.text(0.5 * (moment.start_s + moment.end_s), 1.2, moment.type, ha='center', va='center', fontsize=8, color='black')
ax.set_xlim(0, manifest.duration_s)
ax.set_ylim(0, 1.5)
ax.set_yticks([])
ax.set_xlabel('song time (s)')
ax.set_title('per-note pct_in_tune + highlight regions')
plt.show()

## Reference vs user STARS techniques

STARS techniques aggregated per note window. Reference techniques are blue; matched/added user techniques are green/purple; missed reference techniques are orange.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
for n, comp in zip(reference.notes, analysis.techniques):
    if comp.matched:
        ax.add_patch(Rectangle((n.start_s, 0.6), n.end_s - n.start_s, 0.3, color='#5cd0a5'))
    if comp.user_added:
        ax.add_patch(Rectangle((n.start_s, 0.3), n.end_s - n.start_s, 0.3, color='#c98cf6'))
    if comp.missed:
        ax.add_patch(Rectangle((n.start_s, 0.0), n.end_s - n.start_s, 0.3, color='#f0b35b'))
ax.set_xlim(0, manifest.duration_s)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.15, 0.45, 0.75])
ax.set_yticklabels(['missed', 'user-added', 'matched'])
ax.set_xlabel('song time (s)')
ax.set_title('reference vs user STARS techniques (per note)')
plt.show()

## Coaching cards

The same `CoachingMoment` list the FastAPI UI consumes.

In [ ]:
for m in analysis.highlights.moments:
    print(f'{m.type:>22} | {m.start_s:6.2f}–{m.end_s:6.2f}s | score {m.score:.2f}')
    print(f'  {m.title}')
    print(f'  {m.summary}')
    if m.detail:
        keys = ', '.join(f'{k}={v}' for k, v in m.detail.items() if isinstance(v, (int, float, str)))
        print(f'  ({keys})')
    print()